# Regularization theory — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Adding lambda*||w||^2 shrinks the model toward simplicity
Ridge regression trades fit for a smaller weight norm via a penalty strength lambda. These five examples show the norm shrinking, the coefficient path, the norm collapsing to zero, training error rising as we over-shrink, and variance falling across resamples.

In [ ]:
# 1) the weight norm shrinks with lambda
X=np.array([[1.,1.],[1.,2.],[1.,3.]]); y=np.array([1.,2.,2.])
def ridge(lam): return np.linalg.solve(X.T@X+lam*np.eye(2),X.T@y)
lams=[0,1,10]; plt.figure(figsize=(6,3)); plt.bar([f'lam={l}' for l in lams],[ridge(l)@ridge(l) for l in lams])
plt.ylabel('||w||^2'); plt.title('norm shrinks with lambda'); assert abs(ridge(0)@ridge(0)-0.694)<1e-3

In [ ]:
# 2) the ridge coefficient path
lp=np.logspace(-2,3,60); P=np.array([ridge(l) for l in lp])
plt.figure(figsize=(6,3)); plt.semilogx(lp,P[:,0],label='intercept'); plt.semilogx(lp,P[:,1],label='slope'); plt.axhline(0,c='gray',lw=.5)
plt.legend(); plt.xlabel('lambda'); plt.ylabel('weight'); plt.title('weights shrink toward 0'); assert P[-1]@P[-1]<P[0]@P[0]

In [ ]:
# 3) the norm collapses toward zero
plt.figure(figsize=(6,3)); plt.semilogx(lp,[ridge(l)@ridge(l) for l in lp]); plt.xlabel('lambda'); plt.ylabel('||w||^2')
plt.title('||w||^2 -> 0 as lambda -> infinity'); assert ridge(1000)@ridge(1000)<0.01

In [ ]:
# 4) over-shrinking raises training error (underfitting)
tr=[np.mean((X@ridge(l)-y)**2) for l in lp]; plt.figure(figsize=(6,3)); plt.semilogx(lp,tr)
plt.xlabel('lambda'); plt.ylabel('training MSE'); plt.title('too much lambda underfits'); assert tr[-1]>tr[0]

In [ ]:
# 5) larger lambda lowers variance across resamples
rng=np.random.default_rng(11); xx=np.array([1.,2,3]); plt.figure(figsize=(6,3))
for lam,c in [(0,'tab:blue'),(10,'tab:orange')]:
    preds=[]
    for _ in range(40):
        yy=np.array([1.,2,2])+rng.normal(0,0.3,3)
        w=np.linalg.solve(X.T@X+lam*np.eye(2),X.T@yy); preds.append(X@w)
    preds=np.array(preds); plt.plot(xx,preds.mean(0),'-o',color=c,label=f'lam={lam}')
    plt.fill_between(xx,preds.mean(0)-preds.std(0),preds.mean(0)+preds.std(0),alpha=0.2,color=c)
plt.legend(); plt.title('larger lambda -> lower variance'); assert True